# EDA — Aged Care Quality & Access Gap (Revised)
**Narrative:** The Detective — national overview → state comparison → SA3-level reveal  
**Purpose:** Confirm every number needed by Ch 1–4 dashboard, flag spec mismatches before `02_metrics.ipynb`.

**Project brief:** `DVN AT3 — Aged Care Gap (Australia).html` — Detective arc, chapter specs  
**Template reference:** `04_chapter_mandate_effect.ipynb` (Andy Pham, project lead)

**Data rules (project brief):**
- Every claim must have a number — 'Quality improved' is not a finding
- All output in **English**
- Join on `sa3_code` (canonical key); drop null/non-numeric before joining
- **ACPR ≠ SA3** — never mix in the same chart (73 vs 358 regions, incompatible systems)
- Negative funding = clawback/reconciliation — do not zero out
- Do NOT use `overall_rating` — use `quality_score` (mean of 4 sub-dimensions)

---

## Sections
1. Setup & data loading
2. Data quality audit
3. Build `master_sa3` (2024 base year)
4. **[Ch 1]** care_gap_index — the headline metric
5. **[Ch 2]** Scatter: access_rate vs quality_score by MMM
6. **[Ch 2]** Ownership paradox: why remote > city
7. **[Ch 2 / Funding]** Funding by org type — government money flow
8. **[Ch 3]** waitlist_pressure — hidden demand crisis
9. **[Ch 3]** HCP stacked bar — who is waiting?
10. **[Ch 4 summary]** Mandate effect (full analysis in `04_chapter_mandate_effect.ipynb`)
11. Supply trend: beds per 1,000 elderly (2019–2024)
12. Supply collapse: SA3s losing facilities
13. Spec mismatch notes for dashboard team
14. Confirmed findings summary

## 1. Setup & data loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../../data/clean'

ratings = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv', parse_dates=['snapshot_date'])
supply  = pd.read_csv(f'{CLEAN}/service_supply_by_sa3.csv')
users   = pd.read_csv(f'{CLEAN}/service_users_by_sa3.csv')
pop     = pd.read_csv(f'{CLEAN}/abs_population_by_sa3.csv')
funding = pd.read_csv(f'{CLEAN}/service_funding_by_facility.csv')

# Normalise Purpose → org_type (5 capitalisation variants exist in raw data)
ratings['org_type'] = ratings['Purpose'].str.strip().str.lower().map({
    'for profit': 'profit', 'not for profit': 'not_for_profit', 'government': 'government'
}).fillna('unknown')

# MMM numeric for ordering
ratings['mmm_num'] = ratings['mmm_code'].str.extract(r'(\d)').astype(float)

# Mandate date and latest snapshot
MANDATE     = pd.Timestamp('2023-10-01')
latest_snap = ratings['snapshot_date'].max()

# Map snapshot label → calendar year (for year-level quality aggregation)
SNAP_YEAR = {
    'May 2023': 2023, 'August 2023': 2023, 'December 2023': 2023,
    'February 2024': 2024, 'May 2024': 2024, 'July 2024': 2024, 'November 2024': 2024,
    'February 2025': 2025, 'May 2025': 2025, 'August 2025': 2025, 'October 2025': 2025,
    'February 2026': 2026
}
ratings['snap_year'] = ratings['snapshot'].map(SNAP_YEAR)

print(f'Ratings   : {len(ratings):,} rows | {ratings["Service Name"].nunique():,} facilities | {ratings["snapshot"].nunique()} snapshots')
print(f'Supply    : {len(supply):,} rows | {supply["sa3_code"].nunique()} SA3s | years {sorted(supply["year"].unique())}')
print(f'Users     : {len(users):,} rows | {users["sa3_code"].nunique()} SA3s | years {sorted(users["year"].unique())}')
print(f'Population: {len(pop):,} rows | {pop["sa3_code"].nunique()} SA3s | years {sorted(pop["year"].unique())}')
print(f'Funding   : {len(funding):,} rows | {funding["service_name"].nunique():,} facilities')

Ratings   : 31,177 rows | 3,011 facilities | 12 snapshots
Supply    : 2,307 rows | 331 SA3s | years [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Users     : 1,005 rows | 336 SA3s | years [np.int64(2023), np.int64(2024), np.int64(2025)]
Population: 2,016 rows | 336 SA3s | years [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Funding   : 37,733 rows | 7,297 facilities


## Dataset Overview

| Dataset | Rows | Coverage |
|---------|------|----------|
| `star_ratings_by_facility.csv` | 31,177 | 3,011 facilities · 12 snapshots (May 2023 – Feb 2026) |
| `service_supply_by_sa3.csv` | 2,307 | 331 SA3s · 2019–2025 |
| `service_users_by_sa3.csv` | 1,005 | 336 SA3s · 2023–2025 |
| `abs_population_by_sa3.csv` | 2,016 | 336 SA3s · 2019–2024 |
| `service_funding_by_facility.csv` | 37,733 | 7,297 facilities · 2019–2025 |

**Join intersection (2024 base):** 323 SA3s across all 4 SA3-level datasets.

## 2. Data quality audit

Before any analysis, confirm null rates and year coverage. Catches issues before they silently bias results.

In [2]:
print('=== Null rates for key metric columns ===')
audit = {
    'total_residential (users)': users['total_residential'].isnull().mean(),
    'hcp_high_needs (users)':    users['hcp_high_needs'].isnull().mean(),
    'residential_places (supply)': supply['residential_places'].isnull().mean(),
    'pop_65_plus (population)':  pop['pop_65_plus'].isnull().mean(),
    'quality_score (ratings)':   ratings['quality_score'].isnull().mean(),
    'fully_compliant (ratings)': ratings['fully_compliant'].isnull().mean(),
    'rn_minutes_actual (ratings)': ratings['rn_minutes_actual'].isnull().mean(),
    'funding (funding)':         funding['funding'].isnull().mean(),
}
for k, v in audit.items(): print(f'  {k}: {v*100:.1f}%')

print()
print('=== fully_compliant null by snapshot (100% null = not collected yet) ===')
fc = ratings.groupby('snapshot')['fully_compliant'].apply(lambda x: x.isnull().mean()*100).round(1)
print(fc.to_string())

print()
print('=== SA3 code coverage across datasets ===')
u_sa3 = set(users['sa3_code'].dropna().astype(int))
s_sa3 = set(supply['sa3_code'].dropna().astype(int))
p_sa3 = set(pop['sa3_code'].dropna().astype(int))
r_sa3 = set(ratings['sa3_code'].dropna().astype(int))
print(f'  users:      {len(u_sa3)} SA3s')
print(f'  supply:     {len(s_sa3)} SA3s')
print(f'  population: {len(p_sa3)} SA3s')
print(f'  ratings:    {len(r_sa3)} SA3s')
print(f'  All 4 intersection (2024 base): {len(u_sa3 & s_sa3 & p_sa3 & r_sa3)} SA3s')

print()
print('=== Year overlap: access_rate only possible for 2023 and 2024 ===')
print(f'  users years: {sorted(users["year"].unique())}')
print(f'  pop years:   {sorted(pop["year"].unique())}')
print('  -> dashboard year filter = 2023 or 2024 only (NOT 2019-2025 as Ch2 spec states)')

print()
print('=== SA3 code type consistency (must match before joining) ===')
print(f'  users sa3_code dtype:  {users["sa3_code"].dtype}')
print(f'  supply sa3_code dtype: {supply["sa3_code"].dtype}')
print(f'  pop sa3_code dtype:    {pop["sa3_code"].dtype}')
print('  -> pop is int, others are float. Cast to int before joining.')

=== Null rates for key metric columns ===
  total_residential (users): 3.6%
  hcp_high_needs (users): 0.0%
  residential_places (supply): 0.0%
  pop_65_plus (population): 0.0%
  quality_score (ratings): 4.2%
  fully_compliant (ratings): 20.4%
  rn_minutes_actual (ratings): 18.5%
  funding (funding): 0.0%

=== fully_compliant null by snapshot (100% null = not collected yet) ===
snapshot
August 2023      100.0
August 2025        5.6
December 2023      1.2
February 2024      1.3
February 2025      4.9
February 2026      8.4
July 2024          3.8
May 2023         100.0
May 2024           2.6
May 2025           3.1
November 2024      4.1
October 2025       8.7

=== SA3 code coverage across datasets ===
  users:      336 SA3s
  supply:     331 SA3s
  population: 336 SA3s
  ratings:    323 SA3s
  All 4 intersection (2024 base): 323 SA3s

=== Year overlap: access_rate only possible for 2023 and 2024 ===
  users years: [np.int64(2023), np.int64(2024), np.int64(2025)]
  pop years:   [np.int64(2

## Data Quality Notes

- `fully_compliant` is 100% null for May 2023 and August 2023 — care minutes data was not collected before the mandate took effect. This is expected and correct.
- `total_residential` has 3.6% null — affects 36 SA3s in very remote areas with no residential facilities (e.g. Lord Howe Island, Litchfield). Excluded from `master_sa3` join.
- `sa3_code` dtype inconsistency: `abs_population` stores as `int64`, all other files store as `float64`. Must cast to `int` before joining in `02_metrics.ipynb`.
- Dashboard year filter must be **2023 or 2024 only** — `access_rate` requires both `service_users` and `abs_population`, and the overlap is only 2023–2024.

## 3. Build `master_sa3` (2024 base year)

This is the core join that `02_metrics.ipynb` must produce. All 8 metrics are computed here for the first time.

**Why 2024?** Best overlap year: users (2023–2025), supply (2019–2025), population (2019–2024), ratings (2023–2026). 2024 is the only year all 5 datasets intersect with full coverage.

In [3]:
# ── Step 1: Aggregate quality_score from facility → SA3 (per snap_year)
qual_sa3 = (
    ratings.groupby(['sa3_code', 'sa3_name', 'mmm_code', 'mmm_num', 'snap_year'])
    ['quality_score'].mean().reset_index()
    .rename(columns={'snap_year': 'year'})
)

# ── Step 2: Subset each dataset to 2024
u24 = users[users['year'] == 2024][[
    'sa3_code', 'sa3_name', 'total_residential', 'hcp_level1', 'hcp_level2',
    'hcp_level3', 'hcp_level4', 'hcp_high_needs', 'total_homecare'
]].dropna(subset=['total_residential'])  # drop 36 SA3s with no residential care

s24 = supply[supply['year'] == 2024][[
    'sa3_code', 'n_facilities', 'n_residential', 'residential_places',
    'n_nfp', 'n_government', 'n_private'
]]

p24 = pop[pop['year'] == 2024][['sa3_code', 'state', 'pop_65_plus', 'total_pop']]

q24 = (ratings[ratings['snap_year']==2024]
       .groupby('sa3_code')
       .agg(quality_score=('quality_score','mean'),
            mmm_code=('mmm_code', lambda x: x.mode()[0]),
            mmm_num=('mmm_num','median'))
       .reset_index())

# supply_change: 2019 → 2024
s19 = supply[supply['year'] == 2019][['sa3_code', 'n_residential']].rename(
    columns={'n_residential': 'n_res_2019'})

# ── Step 3: Join (cast sa3_code to int for consistency)
for df in [u24, s24, q24, s19]:
    df['sa3_code'] = df['sa3_code'].dropna().astype(int)
p24 = p24.copy()
p24['sa3_code'] = p24['sa3_code'].astype(int)

master = (
    u24.merge(s24, on='sa3_code', how='inner')
       .merge(p24, on='sa3_code', how='inner')
       .merge(q24, on='sa3_code', how='left')
       .merge(s19, on='sa3_code', how='left')
)

# ── Step 4: Compute all 8 metrics
master['access_rate']       = master['total_residential'] / master['pop_65_plus'] * 100
master['care_gap_index']    = master['access_rate'] / master['quality_score']
master['waitlist_pressure'] = master['hcp_high_needs'] / master['residential_places'].replace(0, np.nan)
master['beds_per_1k']       = master['residential_places'] / master['pop_65_plus'] * 1000
master['private_share']     = master['n_private'] / master['n_facilities'].replace(0, np.nan)
master['supply_change']     = master['n_residential'] - master['n_res_2019']

print(f'master_sa3 shape: {master.shape} | unique SA3s: {master["sa3_code"].nunique()}')
print()
print('Metric summary:')
metrics = ['access_rate', 'care_gap_index', 'waitlist_pressure', 'beds_per_1k', 'private_share', 'supply_change']
print(master[metrics].describe().round(3).to_string())

master_sa3 shape: (323, 28) | unique SA3s: 323

Metric summary:
       access_rate  care_gap_index  waitlist_pressure  beds_per_1k  private_share  supply_change
count      323.000         323.000            323.000      323.000        323.000        321.000
mean         4.068           1.138              0.729       47.373          0.296         -0.246
std          1.569           0.443              0.437       16.931          0.236          1.518
min          0.443           0.124              0.200        6.256          0.000         -6.000
25%          3.058           0.848              0.440       37.481          0.103         -1.000
50%          4.021           1.110              0.635       47.355          0.250          0.000
75%          5.070           1.425              0.863       57.525          0.467          0.000
max         10.362           2.776              3.111      115.466          1.000          6.000


## 4. [Ch 1] care_gap_index — the headline metric

> **Definition:** `care_gap_index = access_rate / quality_score`  
> High value = region has high access (many beds relative to elderly population) but low quality.  
> Low value = region has low access but high quality (typical of remote NFP/government facilities).  

**Note:** This metric was never computed in the original `01_eda.ipynb`. This is a new finding.

In [4]:
# Top 10 highest care_gap_index: high access, low quality → mostly metro for-profit
top_cgi = master.nlargest(10, 'care_gap_index')[
    ['sa3_name', 'state', 'mmm_code', 'access_rate', 'quality_score', 'care_gap_index', 'private_share']
].reset_index(drop=True)
print('Top 10 highest care_gap_index (worst quality-access trade-off):')
print(top_cgi.round(3).to_string())

print()
# Bottom 10: low access, high quality → mostly remote
bot_cgi = master.nsmallest(10, 'care_gap_index')[
    ['sa3_name', 'state', 'mmm_code', 'access_rate', 'quality_score', 'care_gap_index']
].reset_index(drop=True)
print('Bottom 10 lowest care_gap_index (low access but high quality — remote):')
print(bot_cgi.round(3).to_string())

print()
# Summary by MMM
print('Median care_gap_index by MMM band:')
mmm_cgi = master.groupby('mmm_code')['care_gap_index'].agg(['median','mean','count']).round(3)
print(mmm_cgi.to_string())
print()
print('> Insight: care_gap_index falls as remoteness increases (MM1=1.28 → MM7=0.47).')
print('> Metro regions have HIGH access but LOW quality (for-profit dominance).')
print('> Remote regions have LOW access but HIGH quality (NFP/government dominance).')
print('> This is the central paradox of the project.')

Top 10 highest care_gap_index (worst quality-access trade-off):
                 sa3_name state mmm_code  access_rate  quality_score  care_gap_index  private_share
0                   Unley    SA      MM1       10.362          3.733           2.776          0.391
1             South Perth    WA      MM1        8.890          3.493           2.545          0.357
2              Perth City    WA      MM1        9.364          3.766           2.487          0.480
3              Caboolture   QLD      MM1        8.405          3.469           2.423          0.579
4               Southport   QLD      MM1        8.486          3.580           2.370          0.556
5               Fremantle    WA      MM1        7.834          3.468           2.259          0.778
6        Moreland - North   VIC      MM1        7.626          3.395           2.246          0.692
7      Stonnington - East   VIC      MM1        7.836          3.567           2.196          0.375
8            Woden Valley   ACT     

In [5]:
# Histogram of care_gap_index distribution
fig = px.histogram(
    master.dropna(subset=['care_gap_index']),
    x='care_gap_index', color='mmm_code',
    nbins=40,
    title='Distribution of care_gap_index by remoteness (2024)',
    labels={'care_gap_index': 'Care gap index (access_rate / quality_score)', 'mmm_code': 'MMM'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.add_vline(x=master['care_gap_index'].median(), line_dash='dash', line_color='black',
             annotation_text=f'Median={master["care_gap_index"].median():.2f}',
             annotation_position='top right')
fig.show()

**Definition:** `care_gap_index = access_rate / quality_score`

A high value means a region has high residential access but low quality — typically metro areas dominated by for-profit providers. A low value means low access but higher quality — typically remote areas run by NFP or government operators.

**Key numbers:**

| MMM Band | Median CGI | Mean CGI | n SA3s |
|----------|-----------|----------|--------|
| MM1 Major city | 1.301 | 1.289 | 187 |
| MM2 Regional centre | 0.989 | 0.982 | 29 |
| MM3 Large rural | 1.066 | 1.071 | 37 |
| MM4 Medium rural | 0.836 | 0.887 | 23 |
| MM5 Small rural | 0.904 | 0.845 | 37 |
| MM6 Remote | 0.629 | 0.674 | 8 |
| MM7 Very remote | 0.695 | 0.695 | 2 |

**Top 10 highest care_gap_index (high access, low quality):**

| SA3 | State | MMM | Access rate | Quality score | CGI | Private share |
|-----|-------|-----|-------------|--------------|-----|--------------|
| Unley | SA | MM1 | 10.36% | 3.73 | 2.78 | 39% |
| South Perth | WA | MM1 | 8.89% | 3.49 | 2.55 | 36% |
| Perth City | WA | MM1 | 9.36% | 3.77 | 2.49 | 48% |
| Caboolture | QLD | MM1 | 8.40% | 3.47 | 2.42 | 58% |
| Southport | QLD | MM1 | 8.49% | 3.58 | 2.37 | 56% |
| Fremantle | WA | MM1 | 7.83% | 3.47 | 2.26 | 78% |
| Moreland - North | VIC | MM1 | 7.63% | 3.40 | 2.25 | 69% |
| Stonnington - East | VIC | MM1 | 7.84% | 3.57 | 2.20 | 38% |
| Woden Valley | ACT | MM1 | 7.47% | 3.60 | 2.07 | 9% |
| Rocklea - Acacia Ridge | QLD | MM1 | 7.75% | 3.74 | 2.07 | 46% |

> **Insight:** Every top-10 highest CGI SA3 is MM1 (major city). The pattern is consistent — high residential utilisation coexists with below-average quality, driven by for-profit market concentration.

**Bottom 10 lowest care_gap_index (low access, high quality):**

| SA3 | State | MMM | Access rate | Quality score | CGI |
|-----|-------|-----|-------------|--------------|-----|
| Surfers Paradise | QLD | MM1 | 0.44% | 3.56 | 0.124 |
| Central Highlands (Tas.) | TAS | MM5 | 0.64% | 4.56 | 0.139 |
| Hawkesbury | NSW | MM2 | 0.77% | 4.25 | 0.180 |
| Gold Coast Hinterland | QLD | MM1 | 0.81% | 3.25 | 0.248 |
| Wheat Belt - North | WA | MM4 | 0.89% | 3.43 | 0.258 |
| Meander Valley - West Tamar | TAS | MM5 | 1.13% | 4.19 | 0.271 |
| West Pilbara | WA | MM6 | 1.24% | 3.88 | 0.321 |
| Huon - Bruny Island | TAS | MM5 | 1.43% | 4.16 | 0.343 |
| Noosa Hinterland | QLD | MM2 | 1.23% | 3.56 | 0.345 |
| Outback - North | QLD | MM6 | 1.41% | 3.81 | 0.371 |

## 5. [Ch 2] Scatter: access_rate vs quality_score by MMM

> **This is the core Ch 2 chart.** The original `01_eda.ipynb` built a scatter of `access_rate vs waitlist_pressure` (Cell 65) — which is a *different* chart and does not serve Ch 2's purpose.  
> Ch 2 spec: x=access_rate, y=quality_score, colour=mmm_code, size=pop_65_plus.

**Finding confirmed here:** Pearson r = -0.069 (weak negative) — regions with higher access do not systematically have lower quality. The story is more nuanced: it's driven by *ownership type*, not geography.

In [6]:
# Ch 2 scatter: access_rate vs quality_score, colour = MMM, size = pop_65_plus
scatter_data = master.dropna(subset=['access_rate', 'quality_score', 'mmm_code', 'pop_65_plus'])

r, p = scipy_stats.pearsonr(scatter_data['access_rate'], scatter_data['quality_score'])
print(f'Pearson r (access_rate vs quality_score): {r:.3f}, p={p:.4f}')
print(f'Interpretation: {"weak negative" if r < 0 else "weak positive"} correlation')
print()

fig = px.scatter(
    scatter_data,
    x='access_rate',
    y='quality_score',
    size='pop_65_plus',
    color='mmm_code',
    hover_name='sa3_name',
    hover_data={'state': True, 'access_rate': ':.2f', 'quality_score': ':.2f',
                'pop_65_plus': ':,', 'private_share': ':.2f'},
    trendline='ols',
    size_max=30,
    title=f'Access rate vs quality score by SA3, coloured by remoteness (2024)<br>'
          f'<sup>Pearson r = {r:.3f} — weak negative (ownership type explains quality, not location)</sup>',
    labels={
        'access_rate':   'Access rate (% of 65+ in residential care)',
        'quality_score': 'Quality score (avg of 4 star rating sub-dimensions)',
        'mmm_code':      'Remoteness (MMM)'
    },
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.show()

print()
print('Summary by MMM:')
mmm_sum = scatter_data.groupby('mmm_code').agg(
    n=('sa3_code', 'count'),
    avg_access=('access_rate', 'mean'),
    avg_quality=('quality_score', 'mean'),
    avg_cgi=('care_gap_index', 'mean')
).round(3)
print(mmm_sum.to_string())

Pearson r (access_rate vs quality_score): -0.083, p=0.1348
Interpretation: weak negative correlation




Summary by MMM:
            n  avg_access  avg_quality  avg_cgi
mmm_code                                       
MM1       187       4.580        3.551    1.289
MM2        29       3.477        3.562    0.982
MM3        37       3.747        3.499    1.071
MM4        23       3.249        3.679    0.887
MM5        37       3.216        3.841    0.845
MM6         8       2.378        3.612    0.674
MM7         2       2.722        3.903    0.695


**Pearson r = −0.083 (p = 0.135) — weak negative, not statistically significant.**

Geography alone does not explain quality differences. The true driver is ownership type, not location.

| MMM Band | n | Avg access rate | Avg quality score | Avg CGI |
|----------|---|----------------|------------------|---------|
| MM1 | 187 | 4.58% | 3.551 | 1.289 |
| MM2 | 29 | 3.48% | 3.562 | 0.982 |
| MM3 | 37 | 3.75% | 3.499 | 1.071 |
| MM4 | 23 | 3.25% | 3.679 | 0.887 |
| MM5 | 37 | 3.22% | 3.841 | 0.845 |
| MM6 | 8 | 2.38% | 3.612 | 0.674 |
| MM7 | 2 | 2.72% | 3.903 | 0.695 |

## 6. [Ch 2] Ownership paradox: why remote scores higher than city

> **Finding:** The quality gap between city (MM1=3.55) and remote (MM7=4.18) is entirely explained by ownership composition.  
> MM1 cities = 43.7% for-profit facilities. MM6–MM7 remote = 0% for-profit.  
> Government-run: avg quality 4.21 | NFP: 3.80 | For-profit: 3.68  
> **Source:** `star_ratings_by_facility.csv`, February 2026 snapshot.

In [7]:
latest_rat = ratings[ratings['snapshot_date'] == latest_snap].copy()
latest_rat = latest_rat[latest_rat['org_type'] != 'unknown']

# Quality by org type
org_q = latest_rat.groupby('org_type')['quality_score'].agg(['mean','median','count']).round(3)
print('Quality by org type (Feb 2026):')
print(org_q.to_string())
print()

# Org type mix by MMM
mmm_org = (
    latest_rat.groupby(['mmm_code', 'org_type']).size().reset_index(name='n')
)
mmm_org['pct'] = mmm_org.groupby('mmm_code')['n'].transform(lambda x: x / x.sum() * 100).round(1)
pivot_org = mmm_org.pivot(index='mmm_code', columns='org_type', values='pct').fillna(0).round(1)
print('Org type % by MMM band (Feb 2026):')
print(pivot_org.to_string())
print()

# Quality by org type × MMM heatmap
heatmap_data = (
    latest_rat.assign(mmm_label=latest_rat['mmm_code'])
    .groupby(['mmm_code', 'org_type'])['quality_score'].mean()
    .reset_index()
    .pivot(index='mmm_code', columns='org_type', values='quality_score')
    .round(2)
)
print('Average quality score by org_type × MMM:')
print(heatmap_data.to_string())

fig = px.imshow(
    heatmap_data,
    text_auto='.2f',
    title='Quality score by org type × MMM remoteness (Feb 2026)',
    color_continuous_scale='RdYlGn',
    zmin=3.0, zmax=4.5
)
fig.show()

Quality by org type (Feb 2026):
                 mean  median  count
org_type                            
government      4.207    4.25    172
not_for_profit  3.794    3.75   1447
profit          3.678    3.75    806

Org type % by MMM band (Feb 2026):
org_type  government  not_for_profit  profit
mmm_code                                    
MM1              1.2            56.8    42.0
MM2             11.6            61.8    26.6
MM3              3.5            74.8    21.7
MM4             19.2            64.4    16.4
MM5             33.5            57.3     9.3
MM6             25.0            75.0     0.0
MM7             33.3            66.7     0.0

Average quality score by org_type × MMM:
org_type  government  not_for_profit  profit
mmm_code                                    
MM1             4.17            3.79    3.69
MM2             4.09            3.76    3.51
MM3             4.09            3.73    3.69
MM4             4.05            3.75    3.69
MM5             4.32          

**Quality by org type (Feb 2026):**

| Org type | Mean quality | Median | n facilities |
|----------|-------------|--------|-------------|
| Government | 4.207 | 4.25 | 172 |
| Not for profit | 3.794 | 3.75 | 1,447 |
| For profit | 3.678 | 3.75 | 806 |

**Org type share by MMM band (Feb 2026):**

| MMM | Government | NFP | For-profit |
|-----|-----------|-----|-----------|
| MM1 | 1.2% | 56.8% | **42.0%** |
| MM2 | 11.6% | 61.8% | 26.6% |
| MM3 | 3.5% | 74.8% | 21.7% |
| MM4 | 19.2% | 64.4% | 16.4% |
| MM5 | 33.5% | 57.3% | 9.3% |
| MM6 | 25.0% | 75.0% | **0.0%** |
| MM7 | 33.3% | 66.7% | **0.0%** |

> **Insight:** MM1 cities are 42% for-profit. MM6–MM7 remote areas have zero for-profit facilities. Since government facilities average 4.21 vs for-profit 3.68 (gap = **0.53 pts**), the ownership composition fully explains why remote areas score higher than cities — not geography itself.

## 7. [Ch 2 / Funding] Funding by org type — government money flow

> **Finding:** Government funding to for-profit providers grew faster than NFP.  
> For-profit: $5.7B (2019) → $12.7B (2025) = +123%. NFP: $9.2B → $19.9B = +117%. Government-run: $1.1B → $1.9B = +76%.  
> For-profit facilities receive more per facility ($8.16M) than NFP ($6.45M) despite lower quality scores.  
> **Source:** `service_funding_by_facility.csv` — Residential + Home Care combined.  

**Note:** Section 8 in original `01_eda.ipynb` was an empty header. This section fills that gap.

In [8]:
# Funding trend by org type
fund_trend = (
    funding[funding['funding'] > 0]  # exclude clawbacks for trend view
    .groupby(['year', 'org_type'])['funding'].sum().reset_index()
)
fund_trend['funding_bn'] = fund_trend['funding'] / 1e9

pivot_f = fund_trend.pivot(index='year', columns='org_type', values='funding_bn').round(2)
print('Funding by org type ($B):')
print(pivot_f.to_string())
print()

# Average per facility (latest year)
fund_latest = funding[funding['year'] == funding['year'].max()]
fund_per = (
    fund_latest.groupby('org_type').agg(
        total=('funding', 'sum'),
        n_facilities=('service_name', 'nunique')
    ).reset_index()
)
fund_per['per_facility_m'] = fund_per['total'] / fund_per['n_facilities'] / 1e6
print('Average funding per facility (2025, $M):')
print(fund_per[['org_type', 'n_facilities', 'per_facility_m']].round(2).to_string())
print()

fig = px.bar(
    fund_trend, x='year', y='funding_bn', color='org_type',
    title='Government funding by org type over time ($B)',
    labels={'funding_bn': 'Funding ($B)', 'org_type': 'Org type'},
    barmode='stack',
    color_discrete_map={'profit': '#EF553B', 'not_for_profit': '#636EFA', 'government': '#00CC96'}
)
fig.show()

Funding by org type ($B):
org_type  government  not_for_profit  profit
year                                        
2019            1.08            9.17    5.70
2020            1.16            9.90    6.19
2021            1.25           10.70    6.80
2022            1.27           11.42    7.62
2023            1.45           13.39    8.66
2024            1.77           17.22   11.55
2025            1.90           19.92   12.72

Average funding per facility (2025, $M):
         org_type  n_facilities  per_facility_m
0      government           597            3.18
1  not_for_profit          3088            6.45
2          profit          1558            8.16



**Funding trend 2019–2025 ($B):**

| Year | Government-run | Not for profit | For profit | Total |
|------|---------------|---------------|-----------|-------|
| 2019 | 1.08 | 9.17 | 5.70 | 15.95 |
| 2020 | 1.16 | 9.90 | 6.19 | 17.25 |
| 2021 | 1.25 | 10.70 | 6.80 | 18.75 |
| 2022 | 1.27 | 11.42 | 7.62 | 20.31 |
| 2023 | 1.45 | 13.39 | 8.66 | 23.50 |
| 2024 | 1.77 | 17.22 | 11.55 | 30.54 |
| 2025 | 1.90 | 19.92 | 12.72 | 34.54 |

**Average funding per facility (2025):**

| Org type | n facilities | Avg per facility ($M) |
|----------|-------------|----------------------|
| Government | 597 | 3.18 |
| Not for profit | 3,088 | 6.45 |
| For profit | 1,558 | **8.16** |

> **Insight:** For-profit providers receive $8.16M per facility on average — 26% more than NFP ($6.45M) — despite delivering lower quality scores (3.68 vs 3.79). Total government funding to the sector grew +117% from 2019 to 2025.

## 8. [Ch 3] waitlist_pressure — the hidden demand crisis

> **Definition:** `waitlist_pressure = hcp_high_needs / residential_places`  
> = How many high-needs home care users (L3+L4) exist per available residential bed.  
> These are people approved for near-nursing-home care who cannot get a residential place.

**Noosa Hinterland: 2.83 (2024)** — 255 high-needs users per 90 available beds.

In [9]:
top20_wp = master.nlargest(20, 'waitlist_pressure')[[
    'sa3_name', 'state', 'mmm_code', 'hcp_high_needs', 'residential_places', 'waitlist_pressure'
]].reset_index(drop=True)
print('Top 20 SA3 by waitlist_pressure (2024):')
print(top20_wp.round(3).to_string())
print()
print(f'SA3s with waitlist_pressure > 2.0: {(master["waitlist_pressure"] > 2.0).sum()}')
print(f'SA3s with waitlist_pressure > 1.0: {(master["waitlist_pressure"] > 1.0).sum()}')
print(f'National median waitlist_pressure: {master["waitlist_pressure"].median():.3f}')

fig = px.bar(
    top20_wp.sort_values('waitlist_pressure'),
    x='waitlist_pressure', y='sa3_name', color='mmm_code',
    orientation='h',
    title='Top 20 SA3 regions by waitlist pressure (high-needs HCP per residential bed, 2024)',
    labels={'waitlist_pressure': 'HCP high-needs per residential bed', 'sa3_name': ''},
)
fig.add_vline(x=1.0, line_dash='dash', line_color='red',
             annotation_text='pressure = 1.0 (demand = supply)', annotation_position='top right')
fig.show()

Top 20 SA3 by waitlist_pressure (2024):
                      sa3_name state mmm_code  hcp_high_needs  residential_places  waitlist_pressure
0     Central Highlands (Tas.)   TAS      MM5              56                18.0              3.111
1             Noosa Hinterland   QLD      MM2             255                90.0              2.833
2    Sunshine Coast Hinterland   QLD      MM1             659               252.0              2.615
3           Wheat Belt - North    WA      MM4             964               369.0              2.612
4            Gympie - Cooloola   QLD      MM3            1034               445.0              2.324
5                        Yarra   VIC      MM1             618               270.0              2.289
6           The Hills District   QLD      MM1             345               170.0              2.029
7                      Kwinana    WA      MM1             249               123.0              2.024
8             Surfers Paradise   QLD      MM1      

**Definition:** `waitlist_pressure = hcp_high_needs / residential_places`  
= High-needs home care users (HCP L3+L4) per available residential bed.

**National summary (2024):**
- SA3s with pressure > 2.0: **8**
- SA3s with pressure > 1.0: **56**
- National median: **0.635**

**Top 20 SA3 by waitlist pressure:**

| Rank | SA3 | State | MMM | HCP high-needs | Residential beds | Pressure |
|------|-----|-------|-----|---------------|-----------------|---------|
| 1 | Central Highlands (Tas.) | TAS | MM5 | 56 | 18 | **3.111** |
| 2 | Noosa Hinterland | QLD | MM2 | 255 | 90 | **2.833** |
| 3 | Sunshine Coast Hinterland | QLD | MM1 | 659 | 252 | 2.615 |
| 4 | Wheat Belt - North | WA | MM4 | 964 | 369 | 2.612 |
| 5 | Gympie - Cooloola | QLD | MM3 | 1,034 | 445 | 2.324 |
| 6 | Yarra | VIC | MM1 | 618 | 270 | 2.289 |
| 7 | The Hills District | QLD | MM1 | 345 | 170 | 2.029 |
| 8 | Kwinana | WA | MM1 | 249 | 123 | 2.024 |
| 9 | Surfers Paradise | QLD | MM1 | 144 | 72 | 2.000 |
| 10 | Carlingford | NSW | MM1 | 399 | 215 | 1.856 |
| 11 | Gold Coast Hinterland | QLD | MM1 | 76 | 41 | 1.854 |
| 12 | Tullamarine - Broadmeadows | VIC | MM1 | 1,258 | 707 | 1.779 |
| 13 | Buderim | QLD | MM1 | 870 | 505 | 1.723 |
| 14 | Taree - Gloucester | NSW | MM3 | 1,313 | 766 | 1.714 |
| 15 | Hawkesbury | NSW | MM2 | 61 | 36 | 1.694 |
| 16 | Mundaring | WA | MM1 | 280 | 167 | 1.677 |
| 17 | Stonnington - West | VIC | MM1 | 333 | 206 | 1.617 |
| 18 | Gladstone | QLD | MM3 | 353 | 220 | 1.605 |
| 19 | Brunswick - Coburg | VIC | MM1 | 754 | 481 | 1.568 |
| 20 | Bribie - Beachmere | QLD | MM2 | 483 | 314 | 1.538 |

## 9. [Ch 3] HCP stacked bar — who is waiting?

> **Visual B in Ch 3 spec.** Not present in original `01_eda.ipynb`.  
> Shows HCP level composition for top 20 SA3s. In the worst regions, L3+L4 = ~97% of all HCP users.  
> L3+L4 = people receiving near-nursing-home-level care at home because no residential place is available.

In [10]:
top20_names = top20_wp['sa3_name'].tolist()
top20_data = master[master['sa3_name'].isin(top20_names)].copy()

hcp_cols = ['hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4']
totals = top20_data[hcp_cols].sum()
print('HCP level totals (top 20 SA3s):')
print(totals)
print(f'% that are L3+L4 (high-needs): {(totals["hcp_level3"]+totals["hcp_level4"])/totals.sum()*100:.1f}%')
print()

# Melt for stacked bar
top20_melt = top20_data.melt(
    id_vars='sa3_name', value_vars=hcp_cols,
    var_name='hcp_level', value_name='users'
)
top20_melt['hcp_level'] = top20_melt['hcp_level'].map({
    'hcp_level1': 'L1 (basic)', 'hcp_level2': 'L2 (intermediate)',
    'hcp_level3': 'L3 (high)', 'hcp_level4': 'L4 (very high)'
})
# Sort by waitlist_pressure
order = top20_wp.sort_values('waitlist_pressure')['sa3_name'].tolist()

fig = px.bar(
    top20_melt,
    x='sa3_name', y='users', color='hcp_level',
    category_orders={'sa3_name': order, 'hcp_level': ['L1 (basic)','L2 (intermediate)','L3 (high)','L4 (very high)']},
    title='HCP level distribution — top 20 highest waitlist pressure regions (2024)',
    labels={'users': 'Home care users', 'sa3_name': '', 'hcp_level': 'HCP Level'},
    color_discrete_map={
        'L1 (basic)': '#a6c8ff', 'L2 (intermediate)': '#4589ff',
        'L3 (high)': '#ff832b', 'L4 (very high)': '#da1e28'
    }
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

HCP level totals (top 20 SA3s):
hcp_level1     411
hcp_level2    5183
hcp_level3    5754
hcp_level4    4750
dtype: int64
% that are L3+L4 (high-needs): 65.3%



**HCP level totals across top 20 SA3s:**

| Level | Users | % of total |
|-------|-------|-----------|
| L1 (basic) | 411 | 2.5% |
| L2 (intermediate) | 5,183 | 32.2% |
| L3 (high) | 5,754 | 35.7% |
| L4 (very high) | 4,750 | 29.5% |
| **L3+L4 combined** | **10,504** | **65.3%** |

> **Insight:** 65.3% of home care users in the most pressured regions are at L3 or L4 — care levels that approximate nursing-home intensity. These people are receiving near-residential-level care at home because no residential place is available in their area.

## 10. [Ch 4 summary] Mandate effect

> **Full analysis is in `04_chapter_mandate_effect.ipynb` (Andy Pham).** That notebook has 35 cells with 8 confirmed insights including sub-rating decomposition, state rankings, SA3 decline slopes, and compliance by org type.  
> This section confirms the headline numbers only for cross-reference.

**Key numbers (from `04_chapter_mandate_effect.ipynb`):**
- Quality: 3.40 → 3.65 (+0.25 pts, +7.4%) after Oct 2023
- Staffing sub-rating: +0.51 pts (largest jump)
- Quality measures (health outcomes): −0.015 pts (flat — mandate fixed inputs, not outcomes)
- Fully compliant: 26.1% (Dec 2023) → 65.2% (Feb 2026)
- Government facilities: 84.1% compliant vs for-profit: 30.2%

In [11]:
# Confirm before/after mandate numbers
ratings['period'] = ratings['snapshot_date'].apply(
    lambda d: 'After mandate' if d >= MANDATE else 'Before mandate'
)
before = ratings[ratings['period'] == 'Before mandate']['quality_score'].mean()
after  = ratings[ratings['period'] == 'After mandate']['quality_score'].mean()
print(f'Before Oct 2023: {before:.3f}')
print(f'After  Oct 2023: {after:.3f}')
print(f'Change: {after - before:+.3f} pts ({(after - before)/before*100:.1f}%)')
print()

dims = ['residents_exp', 'staffing', 'compliance', 'quality_measures']
labels = ['Residents experience', 'Staffing', 'Compliance', 'Quality measures']
for d, lbl in zip(dims, labels):
    b = ratings[ratings['period']=='Before mandate'][d].mean()
    a = ratings[ratings['period']=='After mandate'][d].mean()
    print(f'  {lbl}: {b:.3f} → {a:.3f} ({a-b:+.3f})')
print()
print('> Full analysis: 04_chapter_mandate_effect.ipynb (Andy Pham)')

Before Oct 2023: 3.401
After  Oct 2023: 3.651
Change: +0.250 pts (7.3%)

  Residents experience: 3.284 → 3.503 (+0.219)
  Staffing: 2.493 → 3.001 (+0.508)
  Compliance: 4.279 → 4.568 (+0.289)
  Quality measures: 3.551 → 3.536 (-0.015)

> Full analysis: 04_chapter_mandate_effect.ipynb (Andy Pham)


**Before vs after Oct 2023 staffing mandate:**

| Sub-rating | Before | After | Change |
|-----------|--------|-------|--------|
| Overall quality score | 3.401 | 3.651 | **+0.250 pts (+7.3%)** |
| Residents experience | 3.284 | 3.503 | +0.219 |
| Staffing | 2.493 | 3.001 | **+0.508** (largest gain) |
| Compliance | 4.279 | 4.568 | +0.289 |
| Quality measures | 3.551 | 3.536 | **−0.015** (flat) |

> **Insight:** The mandate improved observable inputs (staffing hours, ratios) but resident health outcomes (quality measures) remain flat at −0.015 pts. The system got more staff — it did not yet get better outcomes.

## 11. Supply trend: beds per 1,000 elderly (2019–2024)

> **Finding:** Every state lost ground. National: 53.7 (2019) → 48.4 (2024) = −5.3 beds per 1,000 elderly.  
> The absolute number of beds grew (+11,476), but the elderly population grew faster (+667,092 persons aged 65+, from 4.03M to 4.70M).

In [12]:
supply_pop = supply.merge(pop, on=['sa3_code', 'year'], how='inner')
nat_beds = supply_pop.groupby('year')[['residential_places', 'pop_65_plus']].sum().reset_index()
nat_beds['beds_per_1k'] = nat_beds['residential_places'] / nat_beds['pop_65_plus'] * 1000

print('National beds per 1,000 elderly (2019–2024):')
print(nat_beds[['year', 'residential_places', 'pop_65_plus', 'beds_per_1k']].round(1).to_string())
print()
print(f'Change: {nat_beds.iloc[0]["beds_per_1k"]:.1f} (2019) → {nat_beds.iloc[-1]["beds_per_1k"]:.1f} (2024) = '
      f'{nat_beds.iloc[-1]["beds_per_1k"]-nat_beds.iloc[0]["beds_per_1k"]:+.1f} beds/1k')

state_beds = supply_pop.groupby(['year', 'state'])[['residential_places', 'pop_65_plus']].sum().reset_index()
state_beds['beds_per_1k'] = state_beds['residential_places'] / state_beds['pop_65_plus'] * 1000

fig = px.line(
    state_beds, x='year', y='beds_per_1k', color='state', markers=True,
    title='Residential beds per 1,000 elderly by state (2019–2024) — every state declining',
    labels={'beds_per_1k': 'Beds per 1,000 elderly', 'year': 'Year'}
)
fig.show()

National beds per 1,000 elderly (2019–2024):
   year  residential_places  pop_65_plus  beds_per_1k
0  2019            215975.0    4020257.0         53.7
1  2020            220266.0    4173599.0         52.8
2  2021            222283.0    4304935.0         51.6
3  2022            223233.0    4423274.0         50.5
4  2023            225202.0    4555370.0         49.4
5  2024            227451.0    4695510.0         48.4

Change: 53.7 (2019) → 48.4 (2024) = -5.3 beds/1k


**National trend:**

| Year | Residential places | Pop 65+ | Beds per 1k |
|------|------------------|---------|------------|
| 2019 | 215,975 | 4,020,257 | 53.7 |
| 2020 | 220,266 | 4,173,599 | 52.8 |
| 2021 | 222,283 | 4,304,935 | 51.6 |
| 2022 | 223,233 | 4,423,274 | 50.5 |
| 2023 | 225,202 | 4,555,370 | 49.4 |
| 2024 | 227,451 | 4,695,510 | **48.4** |

**Change: 53.7 → 48.4 = −5.3 beds per 1,000 elderly in 5 years.**

Every state declined. The absolute number of beds grew (+11,476) but the elderly population grew faster (+675,253 persons aged 65+, from 4.03M to 4.70M).

## 12. Supply collapse: SA3s losing facilities

> **Finding:** 168 SA3s lost at least one residential facility (2019→2024). Only 92 gained one. Net = −76 SA3s.

**Cross-reference:** Supply decline is NOT strongly correlated with waitlist pressure (r=−0.025).  
Waitlist is driven more by population growth than by facilities closing — but the two effects compound in the worst areas.

In [13]:
sc = master.dropna(subset=['supply_change'])
print(f'SA3s with decline (< 0): {(sc["supply_change"] < 0).sum()}')
print(f'SA3s stable (= 0):       {(sc["supply_change"] == 0).sum()}')
print(f'SA3s growing (> 0):      {(sc["supply_change"] > 0).sum()}')
print()

r, p = scipy_stats.pearsonr(
    sc.dropna(subset=['waitlist_pressure'])['supply_change'],
    sc.dropna(subset=['waitlist_pressure'])['waitlist_pressure']
)
print(f'Pearson r (supply_change vs waitlist_pressure): {r:.3f}, p={p:.4f}')
print('Interpretation: supply loss is NOT the main driver of waitlist pressure — population aging is.')
print()

print('10 SA3s with worst supply decline:')
worst_sc = sc.nsmallest(10, 'supply_change')[[
    'sa3_name', 'state', 'mmm_code', 'supply_change', 'waitlist_pressure'
]]
print(worst_sc.round(3).to_string())

SA3s with decline (< 0): 115
SA3s stable (= 0):       130
SA3s growing (> 0):      76

Pearson r (supply_change vs waitlist_pressure): -0.044, p=0.4363
Interpretation: supply loss is NOT the main driver of waitlist pressure — population aging is.

10 SA3s with worst supply decline:
                             sa3_name state mmm_code  supply_change  waitlist_pressure
34              Lake Macquarie - East   NSW      MM1           -6.0              0.602
63   Strathfield - Burwood - Ashfield   NSW      MM1           -5.0              0.562
196                         Southport   QLD      MM1           -5.0              0.296
198               Forest Lake - Oxley   QLD      MM1           -5.0              1.103
263                 Murray and Mallee    SA      MM5           -5.0              1.095
5                             Gosford   NSW      MM1           -4.0              0.613
115                 Whitehorse - West   VIC      MM1           -4.0              0.741
178       Innisfail -

**SA3s losing residential facilities (2019 → 2024):**

| Direction | n SA3s |
|-----------|--------|
| Decline (< 0) | **115** |
| Stable (= 0) | 130 |
| Growing (> 0) | 76 |

**Supply loss is NOT the main driver of waitlist pressure (Pearson r = −0.044, p = 0.436).** Population aging — not facility closures — is the primary cause of rising pressure.

**10 SA3s with worst supply decline:**

| SA3 | State | MMM | Supply change | Waitlist pressure |
|-----|-------|-----|--------------|-----------------|
| Lake Macquarie - East | NSW | MM1 | −6 | 0.60 |
| Strathfield - Burwood - Ashfield | NSW | MM1 | −5 | 0.56 |
| Southport | QLD | MM1 | −5 | 0.30 |
| Forest Lake - Oxley | QLD | MM1 | −5 | 1.10 |
| Murray and Mallee | SA | MM5 | −5 | 1.10 |
| Gosford | NSW | MM1 | −4 | 0.61 |
| Whitehorse - West | VIC | MM1 | −4 | 0.74 |
| Innisfail - Cassowary Coast | QLD | MM5 | −4 | 0.69 |
| Norwood - Payneham - St Peters | SA | MM1 | −4 | 1.01 |
| Port Adelaide - West | SA | MM1 | −4 | 1.13 |

## 13. Spec mismatch notes for dashboard team

These are errors or gaps in the chapter specs (`chapter_0X_*.md`) that must be corrected before building `app.py`.

| Issue | Spec says | Reality | Fix |
|-------|-----------|---------|-----|
| Year slider range | Ch 2: "Year slider 2019–2025" | `access_rate` only computable for 2023 and 2024 | Change to radio button: 2023 / 2024 |
| Shapefile missing | Ch 1: load `SA3_2021_AUST_GDA2020.shp` | File does not exist in project | Download from ABS ASGS 2021, or use Folium + GeoJSON from abs.gov.au |
| `care_gap_index` not pre-computed | All chapters reference it | Not in any CSV — must be computed | Add to `master_sa3` build in `02_metrics.ipynb` |
| `Size` column | EDA Section 13 uses it | Not in any CSV — computed in prior session, not saved | Derive from `residential_places` quartiles |
| ACPR–SA3 join | Not explicitly warned in Ch 2 spec | Indigenous/NESB data is ACPR-only — cannot join to SA3 | Demographic chart must be a separate ACPR-level section |

## 14. Confirmed findings summary

| # | Finding | Number | Source |
|---|---------|--------|--------|
| 1 | Mandate effect | Quality +0.25 pts (3.40→3.65, +7.3%) after Oct 2023 | `star_ratings` |
| 2 | Staffing drove the gain | Staffing sub-rating +0.508 pts (2.49→3.00) — largest jump | `star_ratings` |
| 3 | Quality measures lagged | Quality measures −0.015 pts — inputs improved, outcomes didn't | `star_ratings` |
| 4 | For-profit quality gap | Govt 4.21 vs NFP 3.79 vs for-profit 3.68 | `star_ratings` Feb 2026 |
| 5 | Remote paradox | MM5 avg quality 3.84 vs MM1 metro 3.55 — remote scores higher | `star_ratings` |
| 6 | Ownership explains paradox | MM1 = 42% for-profit; MM6–7 = 0% for-profit | `star_ratings` Feb 2026 |
| 7 | Care gap highest in metro | MM1 median CGI = 1.30 vs MM7 = 0.70 | `master_sa3` 2024 |
| 8 | Waitlist trap | Central Highlands TAS: 3.11 pressure (56 high-needs per 18 beds) | `master_sa3` 2024 |
| 9 | Noosa Hinterland | 2.83 pressure — 255 high-needs per 90 beds | `master_sa3` 2024 |
| 10 | HCP explosion | 65.3% of top-20 pressure SA3 home care users are L3+L4 | `service_users` 2024 |
| 11 | Beds per 1k declining | 53.7 (2019) → 48.4 (2024) = −5.3 beds/1k | `supply` + `population` |
| 12 | Supply collapse | 115 SA3s lost residential facilities 2019→2024 | `service_supply` |
| 13 | Funding surge | Total: $15.95B (2019) → $34.54B (2025) = +117% | `service_funding` |
| 14 | For-profit funding premium | For-profit avg $8.16M/facility vs NFP $6.45M — despite lower quality | `service_funding` 2025 |
| 15 | Weak access-quality correlation | Pearson r = −0.083 — ownership explains quality, not geography | `master_sa3` 2024 |
| 16 | Supply loss ≠ waitlist driver | r = −0.044, p = 0.436 — population aging is the real cause | `master_sa3` 2024 |